# 03. Bridging GPU Triton to TPU v6e Custom Kernels (`torch.tpu.pallas`)

> **Reference Docs**: [TorchTPU Custom Kernels](https://torchtpu.readthedocs.io/en/0.2.0/custom_kernels/) · [API Reference (`torch.tpu.ops` & `torch.tpu.pallas`)](https://torchtpu.readthedocs.io/en/0.2.0/api_reference/#tpu-custom-operators-torchtpuops-and-pallas-torchtpupallas)

이 노트북에서는 JAX 생태계로 넘어가지 않고 **순수 PyTorch 워크로드 안에서 TPU v6e (Trillium) 하드웨어 커널을 통합**하는 방법을 다룹니다:
1. **GPU Triton vs TPU Pallas 멘탈 모델 매핑**: 1D 포인터 산술(SIMT)에서 2D `(8, 128)` 정렬 `Ref` + `BlockSpec` VMEM 파이프라인으로 사고 전환하기
2. **커스텀 Fused Residual + RMSNorm 커널 작성**: HBM 왕복 트래픽을 단일 VMEM 패스(`block_m=256`)로 압축하고 `torch.tpu.pallas.jax_op`으로 PyTorch 오퍼레이터 등록하기
3. **`torch.compile(backend="tpu")` 그래프 통합 및 벤치마크**: Eager vs Compiled PyTorch vs Custom Pallas Kernel 실효 대역폭(GB/s) 비교

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch_tpu") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "setuptools<81"])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch_tpu", "jax", "xprof", "tensorboard", "tensorboard-plugin-profile", "matplotlib",
    ])
    print("설치 완료! Colab 상단 [런타임 -> 세션 다시 시작] 후 다음 셀부터 실행하세요.")
else:
    print("TorchTPU 환경이 이미 준비되어 있습니다.")

## 1. Mental Model Bridge: GPU Triton → TPU v6e Pallas

| 개념 | GPU `Triton` (`@triton.jit`) | TPU v6e `Pallas` + `TorchTPU` (`torch.tpu.pallas.jax_op`) |
| :--- | :--- | :--- |
| **실행 단위** | `tl.program_id(axis)` (Thread Block / CTA) | `grid=(...)` + `index_map` (TensorCore 타일 반복) |
| **메모리 접근** | `tl.load(ptr + offsets, mask=...)` (임의 1D 포인터) | `BlockSpec(block_shape, index_map)`으로 HBM(~1,638 GB/s) → **VMEM** 비동기 DMA 프리페치 |
| **온칩 연산** | 스칼라/벡터 SIMT 레지스터 | `x_ref[...]`로 VMEM 블록 로드 → **VPU**(`fp32`) / **MXU**($256 \times 256$ `bf16`) |
| **타일 정렬 규칙** | 비교적 자유로움 (2의 거듭제곱 권장) | VPU 레지스터 **`(8, 128)` 배수** 필수 + MXU 연산 시 **`256` 배수** 권장 |
| **PyTorch 연동** | Python wrapper 함수 직접 호출 | `torch.tpu.pallas.jax_op("ns::name", fn)` 등록 (Eager & `torch.compile` 호환) |

In [ ]:
import os
import time
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl
import torch
import torch.nn as nn

os.environ.setdefault("TPU_VISIBLE_DEVICES", "0")
# 주의: TPU 디바이스는 TorchTPU(PjRt)가 단독 점유하도록 torch.device("tpu")로 초기화하며,
# JAX 쪽에서는 직접 디바이스 텐서를 생성하지 않고 커널 정의(Tracing) 용도로만 사용합니다.
device = torch.device("tpu")
print(f"TorchTPU device initialized: {torch.tpu.get_device_properties(device).name}")

## 2. 커스텀 Fused Residual + RMSNorm 커널 (`torch.tpu.pallas.jax_op`)

**최적화 목표**:
- 기본 Eager 실행에서는 `h = x + residual` → `pow(2)` → `mean` → `rsqrt` → `h * rms * weight` 과정에서 HBM을 여러 번 왕복합니다.
- Pallas 커널로 `(block_m=256, D=2048)` 타일(`1 MB`, VMEM 내 여유롭게 상주) 단위로 묶으면:
  1. `x`와 `residual` 타일을 HBM에서 **VMEM**으로 한 번만 읽어오고,
  2. 온칩 **VPU(`fp32`)**에서 잔차 덧셈 + 제곱평균 + 정규화 + 스케일링을 한 번에 수행한 뒤,
  3. 최종 `bf16` 결과만 HBM으로 내보내 TPU v6e의 ~1,638 GB/s HBM 대역폭 효율을 극대화합니다.

In [ ]:
def _fused_add_rmsnorm_kernel(x_ref, res_ref, w_ref, o_ref):
    x_f32 = x_ref[...].astype(jnp.float32)
    res_f32 = res_ref[...].astype(jnp.float32)
    w_f32 = w_ref[...].astype(jnp.float32)  # shape: (1, D)

    h = x_f32 + res_f32
    rms = jax.lax.rsqrt(jnp.mean(jnp.square(h), axis=-1, keepdims=True) + 1e-6)
    out = h * rms * w_f32
    o_ref[...] = out.astype(o_ref.dtype)

def _fused_add_rmsnorm_jax(x: jax.Array, residual: jax.Array, weight_2d: jax.Array) -> jax.Array:
    num_rows, hidden_dim = x.shape
    block_m = 256  # (8, 128) VPU 정렬 및 TPU v6e 256 타일 경계 동시 충족
    assert num_rows % block_m == 0 and hidden_dim % 128 == 0

    return pl.pallas_call(
        _fused_add_rmsnorm_kernel,
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        grid=(num_rows // block_m,),
        in_specs=[
            pl.BlockSpec((block_m, hidden_dim), lambda i: (i, 0)),
            pl.BlockSpec((block_m, hidden_dim), lambda i: (i, 0)),
            pl.BlockSpec((1, hidden_dim), lambda i: (0, 0)),
        ],
        out_specs=pl.BlockSpec((block_m, hidden_dim), lambda i: (i, 0)),
    )(x, residual, weight_2d)

fused_add_rmsnorm_pallas = torch.tpu.pallas.jax_op(
    "pallas::fused_add_rmsnorm",
    _fused_add_rmsnorm_jax,
)

In [ ]:
# PyTorch 레퍼런스 구현과 수치 정합성(Numerical Parity) 검증
def ref_add_rmsnorm(x: torch.Tensor, residual: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:
    h = x.float() + residual.float()
    rms = torch.rsqrt(h.pow(2).mean(dim=-1, keepdim=True) + 1e-6)
    return (h * rms * weight.float()).to(x.dtype)

N_TOKENS, HIDDEN = 16384, 2048
x_tpu = torch.randn(N_TOKENS, HIDDEN, device=device, dtype=torch.bfloat16)
res_tpu = torch.randn(N_TOKENS, HIDDEN, device=device, dtype=torch.bfloat16)
w_tpu = torch.ones(HIDDEN, device=device, dtype=torch.bfloat16)

out_ref = ref_add_rmsnorm(x_tpu, res_tpu, w_tpu)
out_pallas = fused_add_rmsnorm_pallas(x_tpu, res_tpu, w_tpu.unsqueeze(0))
torch.accelerator.synchronize()

max_diff = (out_ref.float() - out_pallas.float()).abs().max().item()
print(f"Max abs difference (PyTorch Ref vs Pallas Kernel): {max_diff:.3e}")
assert torch.allclose(out_ref, out_pallas, atol=2e-2, rtol=2e-2)

## 3. PyTorch 모듈 & `torch.compile(backend="tpu")` 통합 벤치마크

`torch.tpu.pallas.jax_op`으로 감싼 커널은 일반 `nn.Module` 및 `torch.compile(backend="tpu")` 그래프 내부에서도 그래프 브레이크 없이 그대로 인라이닝됩니다.

In [ ]:
class PallasTransformerLayer(nn.Module):
    def __init__(self, hidden_dim: int = 2048, ffn_dim: int = 8192, use_pallas: bool = True):
        super().__init__()
        self.use_pallas = use_pallas
        self.norm_weight = nn.Parameter(torch.ones(1, hidden_dim))
        self.w1 = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.w2 = nn.Linear(ffn_dim, hidden_dim, bias=False)

    def forward(self, x: torch.Tensor, residual: torch.Tensor) -> torch.Tensor:
        if self.use_pallas:
            h = fused_add_rmsnorm_pallas(x, residual, self.norm_weight)
        else:
            h = ref_add_rmsnorm(x, residual, self.norm_weight)
        return self.w2(torch.nn.functional.silu(self.w1(h)))

def bench_variant(label: str, fn, x_in: torch.Tensor, r_in: torch.Tensor, bytes_moved: int, iters: int = 30):
    with torch.no_grad():
        for _ in range(5):
            _ = fn(x_in, r_in)
        torch.accelerator.synchronize()
        t0 = time.perf_counter()
        for _ in range(iters):
            _ = fn(x_in, r_in)
        torch.accelerator.synchronize()
        dt = (time.perf_counter() - t0) / iters
    eff_gbps = (bytes_moved / dt) / 1e9
    print(f"{label:<36} | {dt*1e3:7.3f} ms | Effective BW: {eff_gbps:6.1f} GB/s")

norm_bytes = 3 * N_TOKENS * HIDDEN * 2
print("=== [A] Fused Add + RMSNorm Microbenchmark (Peak HBM ~1,638 GB/s) ===")
with torch.tpu.set_eager_mode(torch.tpu.EagerMode.DEFER_NEVER):
    bench_variant("Strict Eager (Unfused PyTorch)", lambda a, b: ref_add_rmsnorm(a, b, w_tpu), x_tpu, res_tpu, norm_bytes)
bench_variant("torch.compile (XLA Fused)", torch.compile(lambda a, b: ref_add_rmsnorm(a, b, w_tpu), backend="tpu"), x_tpu, res_tpu, norm_bytes)
bench_variant("Custom Pallas Kernel (jax_op)", lambda a, b: fused_add_rmsnorm_pallas(a, b, w_tpu.unsqueeze(0)), x_tpu, res_tpu, norm_bytes)

print("\n=== [B] End-to-End Block with torch.compile(backend='tpu') ===")
ref_layer = PallasTransformerLayer(HIDDEN, 8192, use_pallas=False).to(device=device, dtype=torch.bfloat16).eval()
pallas_layer = PallasTransformerLayer(HIDDEN, 8192, use_pallas=True).to(device=device, dtype=torch.bfloat16).eval()
pallas_layer.load_state_dict(ref_layer.state_dict())

layer_bytes = norm_bytes + 2 * 2 * HIDDEN * 8192
bench_variant("Compiled Block (Standard PyTorch)", torch.compile(ref_layer, backend="tpu"), x_tpu, res_tpu, layer_bytes)
bench_variant("Compiled Block (with Pallas jax_op)", torch.compile(pallas_layer, backend="tpu"), x_tpu, res_tpu, layer_bytes)

## 4. (Sprint 확장 과제) 직접 해볼 다음 실험

1. **v6e Tile Size Sweep**: `block_m`을 `8`, `64`, `128`, `256`, `512`로 바꿔가며 VMEM 타일 크기와 유효 대역폭(GB/s) 변화를 측정해 보세요.
2. **Built-in TPU Custom Ops 비교**: 어텐션 레이어를 추가해 표준 `F.scaled_dot_product_attention`과 TorchTPU 내장 Pallas 커널인 `torch.tpu.ops.splash_sdpa`를 TPU v6e `XProf` Roofline 위에서 비교해 보세요.